In [1]:
import os

folder_name = 'Semantic'

# Crea la cartella se non esiste
os.makedirs(folder_name, exist_ok=True)

print(f"Cartella '{folder_name}' creata o già esistente!")

Cartella 'Semantic' creata o già esistente!


In [8]:
# ==========================================
# 0. INSTALLAZIONE & SETUP
# ==========================================
!pip install -q -U transformers datasets evaluate rouge_score sacrebleu bert-score scikit-learn sentencepiece accelerate

import os
import gc
import json
import zipfile
import torch
import numpy as np
import evaluate
import warnings
from torch.utils.data import DataLoader
from datasets import load_dataset, Value
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    logging as hf_logging
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# --- CONFIGURAZIONI ---
warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["WANDB_DISABLED"] = "true"

def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

def find_hf_model_dir(root_dir: str) -> str:
    root_dir = os.path.abspath(root_dir)
    if os.path.isfile(os.path.join(root_dir, "config.json")):
        return root_dir
    for dirpath, dirnames, filenames in os.walk(root_dir):
        if "config.json" in filenames:
            return dirpath
    raise FileNotFoundError(f"Non trovo config.json sotto: {root_dir}")

def extract_zip(zip_path, extract_to):
    if not os.path.exists(zip_path):
        print(f"ZIP non trovato: {zip_path}")
        return False
    if not os.path.exists(extract_to):
        print(f"Estrazione {os.path.basename(zip_path)}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
    return True

# ==========================================
# 1. PERCORSI
# ==========================================
BASE_DIR = "/content/Semantic"
DATA_MULTI = f"{BASE_DIR}/dataset_multi.json"
DATA_CLF_TEST = f"{BASE_DIR}/dataset_002_multi/test_classification.json"

ZIP_REWRITER = f"{BASE_DIR}/model_inclusive_rewriter_multi.zip"
ZIP_CLASSIFIER = f"{BASE_DIR}/model_classifier_multi.zip"

DIR_REWRITER = "/content/models/rewriter_multi"
DIR_CLASSIFIER = "/content/models/classifier_multi"

extract_zip(ZIP_REWRITER, DIR_REWRITER)
extract_zip(ZIP_CLASSIFIER, DIR_CLASSIFIER)

REAL_RWR_DIR = find_hf_model_dir(DIR_REWRITER)
REAL_CLF_DIR = find_hf_model_dir(DIR_CLASSIFIER)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ==========================================
# 2. METRICHE CLASSIFICATORE (AUTO-DETECT LABEL)
# ==========================================
print("\n" + "="*30)
print("CLASSIFICATORE (MULTI) - CORRECT LABELS")
print("="*30)

if os.path.exists(DATA_CLF_TEST):
    # 1. Carichiamo PRIMA il modello per leggere la sua config
    print("Caricamento configurazione modello...")
    clf_tokenizer = AutoTokenizer.from_pretrained(REAL_CLF_DIR)
    clf_model = AutoModelForSequenceClassification.from_pretrained(REAL_CLF_DIR).to(device)
    clf_model.eval()

    # LEGGIAMO COME IL MODELLO VEDE IL MONDO (id2label)
    id2label = clf_model.config.id2label
    print(f"Configurazione Etichette del Modello: {id2label}")

    # Determiniamo quale ID corrisponde a "Non Inclusiva"
    # Cerchiamo parole chiave nel config del modello
    non_incl_id = 1 # default
    incl_id = 0     # default

    if id2label:
        for k, v in id2label.items():
            val_str = str(v).lower()
            if "non" in val_str or "not" in val_str:
                non_incl_id = int(k)
            elif "incl" in val_str:
                incl_id = int(k)

    print(f"Mapping rilevato: Inclusive -> {incl_id}, Non-Inclusive -> {non_incl_id}")

    # 2. Carichiamo e adattiamo il Dataset
    try:
        dataset = load_dataset("json", data_files={"test": DATA_CLF_TEST})["test"]
    except:
        dataset = load_dataset("json", data_files={"test": DATA_CLF_TEST}, split="test")

    # Funzione di mapping DINAMICA basata sulla config del modello
    def fix_labels(ex):
        val = ex.get("label") or ex.get("label_id") or ex.get("target")
        s_val = str(val).lower().strip()

        # Se nel dataset c'è scritto "non...", assegniamo l'ID che il modello si aspetta per "non..."
        if "non" in s_val or s_val == "1" or s_val == "true":
            ex["label"] = non_incl_id
        else:
            ex["label"] = incl_id
        return ex

    dataset = dataset.map(fix_labels)
    if "sentence" in dataset.column_names:
        dataset = dataset.rename_column("sentence", "text")
    dataset = dataset.cast_column("label", Value("int64"))

    print(f"Dataset caricato: {len(dataset)} esempi.")

    # 3. DataLoader e Predizione
    def clf_collate(batch):
        texts = [str(x["text"]) for x in batch]
        labels = [int(x["label"]) for x in batch]
        enc = clf_tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc

    clf_loader = DataLoader(dataset, batch_size=64, shuffle=False, collate_fn=clf_collate)

    all_preds, all_labels = [], []

    print("Calcolo predizioni...")
    with torch.no_grad():
        for i, batch in enumerate(clf_loader):
            labels = batch.pop("labels").cpu().numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            out = clf_model(**batch)
            probs = torch.softmax(out.logits, dim=-1).detach().cpu().numpy()
            preds = np.argmax(probs, axis=-1)
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    # 4. Metriche
    acc = accuracy_score(all_labels, all_preds)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)

    print(f"\nAccuracy: {acc}")
    print(f"Weighted P/R/F1: {p_w} {r_w} {f1_w}")
    print(f"Macro    P/R/F1: {p_m} {r_m} {f1_m}")

    print("\nConfusion matrix:\n", confusion_matrix(all_labels, all_preds))

    # Nomi target corretti presi dal modello
    try:
        names = [id2label[i] for i in range(len(id2label))]
    except:
        names = ["Classe 0", "Classe 1"]

    print("\nClassification report:\n", classification_report(all_labels, all_preds, target_names=names, zero_division=0))

    del clf_model, clf_tokenizer
    clean_memory()

else:
    print(f"Dataset Classificatore non trovato: {DATA_CLF_TEST}")

# ==========================================
# 3. METRICHE RISCRITTORE (Sample Test)
# ==========================================
print("\n" + "="*30)
print("RISCRITTORE (MULTI) - SAMPLE TEST")
print("="*30)

if os.path.exists(DATA_MULTI):
    ds = load_dataset("json", data_files={"train": DATA_MULTI})
    test_ds = ds["train"].train_test_split(test_size=0.1, seed=42)["test"]
    test_ds = test_ds.select(range(min(100, len(test_ds))))

    src_texts = test_ds["non_inclusiva"]
    ref_texts = test_ds["inclusiva"]

    rwr_tokenizer = AutoTokenizer.from_pretrained(REAL_RWR_DIR, use_fast=False)
    rwr_model = AutoModelForSeq2SeqLM.from_pretrained(REAL_RWR_DIR, torch_dtype=torch.float16).to(device)
    rwr_model.eval()

    rouge = evaluate.load("rouge")
    bleu = evaluate.load("sacrebleu")
    bertscore = evaluate.load("bertscore")

    def batched_generate(texts, batch_size=8):
        preds = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i+batch_size]
            enc = rwr_tokenizer(chunk, truncation=True, padding=True, max_length=128, return_tensors="pt").to(device)
            with torch.no_grad():
                gen = rwr_model.generate(**enc, max_new_tokens=96, num_beams=4, early_stopping=True)
            out = rwr_tokenizer.batch_decode(gen, skip_special_tokens=True)
            preds.extend([o.strip() for o in out])
        return preds

    print(f"Generazione su {len(src_texts)} frasi...")
    pred_texts = batched_generate(src_texts, batch_size=8)

    rouge_res = rouge.compute(predictions=pred_texts, references=ref_texts)
    rouge_res = {k: float(v) * 100 for k, v in rouge_res.items()}
    print("ROUGE (MULTI) [final]:", rouge_res)

    bleu_res = bleu.compute(predictions=pred_texts, references=[[r] for r in ref_texts])
    print("BLEU (SacreBLEU, MULTI) [final]:", bleu_res)

    print("Calcolo BERTScore...")
    bs = bertscore.compute(predictions=pred_texts, references=ref_texts, model_type="xlm-roberta-base", lang="other", device=device, batch_size=8)

    print(f"BERTScore (MULTI) model: xlm-roberta-base")
    print("BERTScore mean P/R/F1 [final]:",
          float(np.mean(bs["precision"])),
          float(np.mean(bs["recall"])),
          float(np.mean(bs["f1"])))

    print("\n--- ESEMPI (MULTI) ---")
    for i in range(min(3, len(src_texts))):
        print(f"\nIN  : {src_texts[i]}")
        print(f"PRED: {pred_texts[i]}")
        print(f"REF : {ref_texts[i]}")

    clean_memory()
else:
    print(f"Dataset Riscrittore non trovato: {DATA_MULTI}")

Device: cuda

CLASSIFICATORE (MULTI) - CORRECT LABELS
Caricamento configurazione modello...
Configurazione Etichette del Modello: {0: 'not_inclusive', 1: 'inclusive'}
Mapping rilevato: Inclusive -> 1, Non-Inclusive -> 0


Map:   0%|          | 0/8644 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/8644 [00:00<?, ? examples/s]

Dataset caricato: 8644 esempi.
Calcolo predizioni...

Accuracy: 0.8779500231374364
Weighted P/R/F1: 0.9037138006990182 0.8779500231374364 0.8795925260699774
Macro    P/R/F1: 0.877502643956081 0.8986687787219887 0.8762504772021309

Confusion matrix:
 [[3288   38]
 [1017 4301]]

Classification report:
                precision    recall  f1-score   support

not_inclusive       0.76      0.99      0.86      3326
    inclusive       0.99      0.81      0.89      5318

     accuracy                           0.88      8644
    macro avg       0.88      0.90      0.88      8644
 weighted avg       0.90      0.88      0.88      8644


RISCRITTORE (MULTI) - SAMPLE TEST
Generazione su 100 frasi...
ROUGE (MULTI) [final]: {'rouge1': 21.506516072043297, 'rouge2': 7.896835761202118, 'rougeL': 19.81100610779582, 'rougeLsum': 19.74744359385093}
BLEU (SacreBLEU, MULTI) [final]: {'score': 6.00178203646565, 'counts': [338, 93, 42, 21], 'totals': [1208, 1108, 1008, 908], 'precisions': [27.980132450331126